# 10 — Train VAE

Short training run using `poregen` library modules.  
Run on an interactive HPC node with GPU(s).

In [1]:
from poregen.configs.config import load_config
from pathlib import Path
from torch.utils.data import DataLoader
import torch

from poregen.dataset.loader import PatchDataset
from poregen.models.vae import build_vae
from poregen.losses import compute_total_loss
from poregen.training import (
    seed_everything,
    select_device,
    get_autocast_dtype,
    make_scaler,
    train_loop,
)

## Config

In [2]:
cfg = load_config("../src/poregen/configs/vae_default.yaml")

# DGX Spark overrides
cfg["data"]["num_workers"] = 12

print(cfg)

{'model': {'name': 'conv_noattn', 'z_channels': 8, 'base_channels': 32, 'n_blocks': 2, 'patch_size': 64}, 'loss': {'xct_loss_type': 'charbonnier', 'xct_weight': 1.0, 'mask_bce_weight': 1.0, 'mask_bce_pos_weight': 17.16, 'mask_dice_weight': 1.0, 'use_tversky': True, 'tversky_alpha': 0.3, 'tversky_beta': 0.7, 'kl_free_bits': 0.0, 'kl_warmup_steps': 0, 'kl_max_beta': 0.05}, 'training': {'seed': 42, 'lr': 0.0002, 'weight_decay': 0.01, 'total_steps': 71690, 'max_grad_norm': None, 'scheduler': 'none', 'lr_min': 2e-05, 'warmup_steps': 0, 'log_every': 1, 'eval_every': 62, 'val_batches': 20, 'test_every': 625, 'test_batches': 20, 'save_every': 1000, 'image_log_every': 62, 'sample_every': 12500, 'n_patch_samples': 8}, 'data': {'batch_size': 128, 'num_workers': 12, 'pin_memory': True}}


## Setup

In [3]:
seed_everything(cfg["training"]["seed"])
device = select_device()  # or select_device(gpu_id=1)
autocast_dtype = get_autocast_dtype(device)
scaler = make_scaler(device)
print(f"Device: {device}  |  AMP dtype: {autocast_dtype}")

Device: cuda:0  |  AMP dtype: torch.bfloat16


## Data

In [4]:
DATA_ROOT = Path("../data/processed")
INDEX     = DATA_ROOT / "patch_index.parquet"
train_ds = PatchDataset(INDEX, DATA_ROOT, split="train")
val_ds   = PatchDataset(INDEX, DATA_ROOT, split="val")
test_ds  = PatchDataset(INDEX, DATA_ROOT, split="test")

train_loader = DataLoader(
    train_ds,
    batch_size=cfg["data"]["batch_size"],
    shuffle=True,
    num_workers=cfg["data"]["num_workers"],
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg["data"]["batch_size"],
    shuffle=False,
    num_workers=cfg["data"]["num_workers"],
    pin_memory=True,
)
test_loader = DataLoader(
    test_ds,
    batch_size=cfg["data"]["batch_size"],
    shuffle=False,
    num_workers=cfg["data"]["num_workers"],
    pin_memory=True,
)

print(f"Train patches: {len(train_ds):,}  |  Val patches: {len(val_ds):,}  |  Test patches: {len(test_ds):,}")

Train patches: 1,835,343  |  Val patches: 230,819  |  Test patches: 232,221


## Model + Optimizer

In [5]:
model = build_vae(
    cfg["model"]["name"],
    z_channels=cfg["model"]["z_channels"],
    base_channels=cfg["model"]["base_channels"],
    n_blocks=cfg["model"]["n_blocks"],
    patch_size=cfg["model"]["patch_size"],
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg["training"]["lr"],
    weight_decay=cfg["training"]["weight_decay"],
)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

Model params: 577,746


In [6]:
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

_t = cfg["training"]
if _t.get("scheduler", "none") == "cosine":
    warmup = LinearLR(
        optimizer,
        start_factor=0.01,
        end_factor=1.0,
        total_iters=_t["warmup_steps"],
    )
    cosine = CosineAnnealingLR(
        optimizer,
        T_max=_t["total_steps"] - _t["warmup_steps"],
        eta_min=_t["lr_min"],
    )
    scheduler = SequentialLR(
        optimizer,
        schedulers=[warmup, cosine],
        milestones=[_t["warmup_steps"]],
    )
    print(f"Scheduler: linear warmup {_t['warmup_steps']} steps → cosine decay to {_t['lr_min']:.0e}")
else:
    scheduler = None
    print("Scheduler: none")

Scheduler: none


## Loss function

In [7]:
loss_fn = lambda output, batch, step: compute_total_loss(output, batch, step, cfg)

In [ ]:
from torch.utils.tensorboard import SummaryWriter
import shutil

m = cfg["model"]
t = cfg["training"]
l = cfg["loss"]
d = cfg["data"]

# Encode every decision that affects training dynamics in the run name.
# Format: {arch}_z{z_ch}_c{base_ch}_bs{bs}_lr{lr}_b{beta}_fb{fb}_klw{klw}_sched{sched}_clip{clip}
EXP_NAME = (
    f"{m['name']}"
    f"_z{m['z_channels']}"
    f"_c{m['base_channels']}"
    f"_bs{d['batch_size']}"
    f"_lr{t['lr']:.0e}"
    f"_b{l['kl_max_beta']:.3f}"
    f"_fb{l['kl_free_bits']}"
    f"_klw{l['kl_warmup_steps']}"
    f"_sched{t['scheduler']}"
    f"_clip{'none' if t['max_grad_norm'] is None else t['max_grad_norm']}"
)
RUN_DIR = Path(f"../runs/vae/{EXP_NAME}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Experiment: {EXP_NAME}")

# ── Persist the exact config used for this run ──────────────────────────
# config.yaml is the ground truth: readable without TensorBoard,
# survives log rotation, and is enough to reproduce the run from scratch.
shutil.copy("../src/poregen/configs/vae_default.yaml", RUN_DIR / "config.yaml")
print(f"Config saved → {RUN_DIR / 'config.yaml'}")

tb_writer = SummaryWriter(str(RUN_DIR / "tb"))

# Also log hparams to TensorBoard HPARAMS tab (secondary, less reliable).
hparams = {
    "arch":            m["name"],
    "z_channels":      m["z_channels"],
    "base_channels":   m["base_channels"],
    "n_blocks":        m["n_blocks"],
    "patch_size":      m["patch_size"],
    "batch_size":      d["batch_size"],
    "lr":              t["lr"],
    "weight_decay":    t["weight_decay"],
    "scheduler":       t["scheduler"],
    "warmup_steps":    t["warmup_steps"],
    "max_grad_norm":   str(t["max_grad_norm"]),
    "kl_max_beta":     l["kl_max_beta"],
    "kl_warmup_steps": l["kl_warmup_steps"],
    "kl_free_bits":    l["kl_free_bits"],
    "xct_loss_type":   l["xct_loss_type"],
    "xct_weight":      l["xct_weight"],
    "mask_bce_weight": l["mask_bce_weight"],
    "mask_dice_weight":l["mask_dice_weight"],
    "use_tversky":     str(l["use_tversky"]),
    "total_steps":     t["total_steps"],
    "seed":            t["seed"],
}
tb_writer.add_hparams(hparams, metric_dict={"hparam/placeholder": 0.0})

history = train_loop(
    model,
    train_loader,
    val_loader,
    optimizer,
    scaler,
    loss_fn,
    total_steps=cfg["training"]["total_steps"],
    log_every=cfg["training"]["log_every"],
    eval_every=cfg["training"]["eval_every"],
    val_batches=cfg["training"]["val_batches"],
    test_loader=test_loader,
    test_every=cfg["training"]["test_every"],
    test_batches=cfg["training"]["test_batches"],
    save_every=cfg["training"]["save_every"],
    image_log_every=cfg["training"]["image_log_every"],
    sample_every=cfg["training"]["sample_every"],
    n_patch_samples=cfg["training"]["n_patch_samples"],
    run_dir=RUN_DIR,
    device=device,
    autocast_dtype=autocast_dtype,
    max_grad_norm=cfg["training"]["max_grad_norm"],
    scheduler=scheduler,
    tb_writer=tb_writer,
)

tb_writer.close()
print(f"Done. Final loss: {history[-1]['total']:.4f}")

Experiment: conv_noattn_z8_c32_bs128_lr2e-04_b0.050_fb0.0_klw0_schednone_clipnone
Config saved → ../runs/vae/conv_noattn_z8_c32_bs128_lr2e-04_b0.050_fb0.0_klw0_schednone_clipnone/config.yaml


Training:  14%|█▎        | 9702/71690 [9:13:47<41:50:24,  2.43s/it, kl=2.6581, loss=0.3075, β=0.0500] 

## Quick loss plot

In [ ]:
import matplotlib.pyplot as plt

train_h = [r for r in history if r["split"] == "train"]
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
ax.plot([r["step"] for r in train_h], [r["total"] for r in train_h], lw=0.6)
ax.set(xlabel="Step", ylabel="Total loss", title="Training loss")
plt.tight_layout()
plt.show()

NameError: name 'history' is not defined